# LLM examples

## Learning goals: 

- Understand the concept and mechanisms of "tools" in the context of LLMs by examples. 

In this notebook, we will manually create a mechanism to call an external "tool" to complement the LLM's output. This is an exercise to understand the prompting strategy. 

(In practice, most LLMs now have systematic ways to specify new tools and each provides a specific syntax to specify tools. Also note that working with tools, and some of the related prompting strategies, are very recent and there is still active research in this). 

## Dependencies

We will use the packages `cohere` (see notebook 1) and `openfoodfacts`. Make sure these are installed. 

## Environment preparation

In [1]:
import cohere
import openfoodfacts

/Users/joseantonio.rodriguez15/Library/CloudStorage/OneDrive-UniversitatRamónLlull/projects-ES-FR62M6XQ1J/teaching_prototyping/consulta/PDAI26/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
with open("cohere.key") as f:
    COHERE_API_KEY = f.read()
cohere_client = cohere.ClientV2(COHERE_API_KEY)
api = openfoodfacts.API(user_agent="MyAwesomeApp/1.0")


## Working with tools

Imagine we ask Cohere about the calories of some food item. While it will still provide an answer, this answer will be based on the word statistics learned by the model. 

In [5]:
prompt = input("Enter some text")

response = cohere_client.chat(
    messages=[{
        "role": "user",
        "content": prompt
    }],
    model="command-a-03-2025"
)

print(response.message.content[0].text)

The number of calories in a box of Pringles can vary depending on the size of the box and the flavor. Here’s a general breakdown for a standard **5.96 oz (169g)** can of Original Pringles:

- **Calories per can**: Approximately **900 calories**.
- **Calories per serving**: A serving size is typically **1 oz (28g)**, which contains about **150 calories**.

For other flavors or sizes, the calorie count may differ slightly. Always check the nutrition label on the specific product for the most accurate information.


Let's say, in order to have a more accurate answer, we want to connect this LLM with the OpenFoodFacts database. 

First, we create a function that queries the database given the name of the product and returns the calories: 

In [14]:
def get_calories(name):
    """
    Get the calories of a product by its name.
    """
    result = api.product.text_search(name)
    print(result)

    # This just takes the first item of the search. 
    # In a real application, you would need to do more error checking.
    first_match = result['products'][0]
    id = first_match['_id']
    result = api.product.get(id, fields=["code", "product_name", "energy-kcal_100g"])
    return result['energy-kcal_100g']
    #return 100


In [15]:
get_calories("Pringles")

{'count': 1295, 'page': 1, 'page_count': 20, 'page_size': 20, 'products': [{'_id': '5053990156009', '_keywords': ['antipasti', 'authority', 'base', 'bevande', 'cibi', 'crisp', 'de', 'di', 'food', 'fritti', 'from', 'frutta', 'halal', 'made', 'man', 'original', 'patatine', 'potato', 'pringle', 'reconstituted', 'salati', 'sale', 'snack', 'sottili', 'tidy', 'triman', 'vegano', 'vegetable', 'vegetale', 'vegetariano', 'verdura', 'verdure'], 'abbreviated_product_name': 'ORG Original 175g', 'abbreviated_product_name_fr': 'ORG Original 175g', 'abbreviated_product_name_fr_imported': 'ORG Original 175g', 'added_countries_tags': [], 'additives_n': 2, 'additives_original_tags': ['en:e471', 'en:e160bii'], 'additives_tags': ['en:e160b', 'en:e160bii', 'en:e471'], 'allergens': 'en:gluten', 'allergens_from_ingredients': 'en:gluten, blé', 'allergens_from_user': '(it) en:gluten', 'allergens_hierarchy': ['en:gluten'], 'allergens_imported': 'Gluten', 'allergens_lc': 'it', 'allergens_tags': ['en:gluten'], 'a

ReadTimeout: HTTPSConnectionPool(host='world.openfoodfacts.org', port=443): Read timed out. (read timeout=10.0)

Now we can use the following strategy for the LLM to call this function if needed. 

- We do a first call to the LLM to check if the prompt asks about calories. If not, respond normally. 
- If the prompt asks about calories, respond with some internal "code" so that we can intercept it and do further processing. 
- If we detect the code, then we call the above function. 
- Finally, we create a new prompt  asking the LLM to provide a final response based on the original request and the information returned by the function. 

In [9]:
message = input("Enter some text")
system_prompt = """
You are a general-purpose conversational assistant.
Respond normally to all user messages except when the user asks for the calorie
content of a food item.
If the user asks for the calories of a specific food item, you must:
- Respond with exactly this answer: CALORIES_REQUEST <food item>
- Replace <food item> with the name of the product mentioned by the user.
- Do not include any additional text, explanation, punctuation, or formatting.
- Do not answer the calorie question.

Example:
User: "How many calories are in an apple?"
Assistant: CALORIES_REQUEST apple

In all other cases, behave like a normal chatbot."""

response = cohere_client.chat(
    messages = [
        {'role':'system', 'content': system_prompt},
        {'role': 'user', 'content': message}
    ],
    model="command-a-03-2025"
)

output = response.message.content[0].text

if output.startswith("CALORIES_REQUEST"):
    product_name = output.split(" ")[1]
    calories = get_calories(product_name)
    print("<INTERMEDIATE RESULT> Calories=", calories)
    final_response = cohere_client.chat(
        messages=[
            {
            'role': 'system',
            'content': f"""
Now complete the request of the user, knowing 
the calories of the product on the OpenFoodFacts database is {calories}.
Mention this source in the response.
            """
            },
            {"role": "user", "content": message}
        ],
        model="command-a-03-2025")
    print("Final response: ", final_response.message.content[0].text)
else:
    print("Final response: ", output)



<INTERMEDIATE RESULT> Calories= 100
Final response:  According to the OpenFoodFacts database, the calories for the product in question (likely referring to a standard serving or a specific size of Pringles) is **100 calories**. However, it's important to note that the calorie content of Pringles can vary depending on the serving size and flavor. A typical serving size for Pringles is about 16 chips, which usually contains around 150 calories. For an entire box (tube) of Pringles, which is approximately 5.96 ounces (169 grams), the total calorie count is generally around **900-1000 calories**, depending on the flavor. Always check the nutrition label on the specific product for the most accurate information. 

*Source: OpenFoodFacts database (for the 100-calorie reference).*
